## Finite-time blow-up (a simplified 2-D demonstration)

The Clay Millennium Prize asks whether the **three-dimensional** incompressible Navier–Stokes equations, starting from smooth divergence-free data, can stay smooth for all time or can develop a singularity in finite time.

In November 2026, Alpöge and Buckmaster posted a claim of **force-free** finite-time blow-up in 3-D, built with heavy assistance from Claude: smooth compactly supported data on $\mathbb{R}^3$ (and smooth data on the torus) whose solution blows up, with the gradient growing like $(T-t)^{-1/4}$ in $\dot H^1$. That construction lives in self-similar variables and a numerical "certificate"; it is **not** what this notebook runs.

What we can show here is a **simplified visual analogue** with the lid-driven cavity solver in this repository:

- the equations are still 2-D Navier–Stokes on a unit square,
- the lid drives a primary vortex,
- with an aggressive time step the explicit scheme loses control and the fields blow up in finite simulation time.

That blow-up is a **numerical** singularity of the discrete scheme (CFL / explicit Euler failure), not a continuum proof of the Millennium problem. The point of the notebook is to watch a coherent vortex form and then fail, and to read the diagnostics the solver already returns when that happens.

### System setup

In [ ]:
import warnings
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.append('../src/')
from flow_simulation import simulate_flow

### Shared cavity

Same grid, Reynolds number, and lid for both runs. Only the time step changes.

In [ ]:
Nxc = Nyc = 48
Re = 200.0
SOR = 1.5
n_iter = 10
Us, Un, Vw, Ve = 0.0, 1.0, 0.0, 0.0

dx = 1.0 / Nxc
print(f"dx = {dx:.4g}")
print(f"convective CFL ~ Un*dt/dx; diffusive limit dt < Re*dx^2/4 = {Re * dx**2 / 4:.4g}")

### Stable control

A smaller `dt` keeps the convective CFL below 1. The cavity forms a primary vortex and `max|u|` stays near the lid speed.

In [ ]:
dt_stable = 0.01
tf_stable = 1.2
u0 = np.zeros((Nxc + 1, Nyc + 2))
v0 = np.zeros((Nxc + 2, Nyc + 1))

with warnings.catch_warnings(record=True) as w_stable:
    warnings.simplefilter("always")
    X, Y, STREAMLINE, u_s, v_s, p_s, vort_s, diag_s = simulate_flow(
        u0, v0, Nxc, Nyc, dt_stable, tf_stable, n_iter, SOR, Re,
        Us, Un, Vw, Ve, return_diagnostics=True, verbose=False,
    )

print(f"CFL = {diag_s['cfl']:.2f}")
print(f"diverged_step = {diag_s['diverged_step']}")
print(f"steps completed = {diag_s['steps_completed']}")
print(f"final max|u| = {diag_s['umax'][-1]:.3g}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5), dpi=100)
ax[0].contour(X, Y, STREAMLINE, 40)
ax[0].set_title("Stable run — streamlines")
ax[0].set_aspect("equal")
im = ax[1].contourf(np.rot90(np.fliplr(vort_s)), 30)
ax[1].set_title("Stable run — vorticity")
ax[1].set_aspect("equal")
fig.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout()

### Blow-up run

Raise `dt` so the convective CFL exceeds 1. The same lid still builds a vortex, then the velocity and vorticity explode and the solver stops, returning the last finite picture.

In [ ]:
dt_blow = 0.025
tf_blow = 2.0

# Keep a few snapshots while the vortex is still healthy, then near the crash.
snapshots = []

def capture(n, t, fields):
    snapshots.append({
        "n": n,
        "t": t,
        "umax": float(np.max(np.abs(fields["u_contour"]))),
        "vort_max": float(np.max(np.abs(fields["vorticity"]))),
        "STREAMLINE": fields["STREAMLINE"].copy(),
        "vorticity": fields["vorticity"].copy(),
        "u_contour": fields["u_contour"].copy(),
        "v_contour": fields["v_contour"].copy(),
    })

u0 = np.zeros((Nxc + 1, Nyc + 2))
v0 = np.zeros((Nxc + 2, Nyc + 1))

with warnings.catch_warnings(record=True) as w_blow:
    warnings.simplefilter("always")
    Xb, Yb, STREAMb, u_b, v_b, p_b, vort_b, diag_b = simulate_flow(
        u0, v0, Nxc, Nyc, dt_blow, tf_blow, n_iter, SOR, Re,
        Us, Un, Vw, Ve,
        return_diagnostics=True, verbose=False,
        callback=capture, callback_every=4,
    )

print(f"CFL = {diag_b['cfl']:.2f}")
print(f"diverged_step = {diag_b['diverged_step']}")
print(f"diverged_time = {diag_b['diverged_time']}")
print(f"steps completed = {diag_b['steps_completed']}")
print(f"last recorded max|u| = {diag_b['umax'][-1]:.3g}")
print()
print("solver warning:")
for msg in w_blow:
    if "diverged" in str(msg.message).lower() or "CFL" in str(msg.message):
        print(" ", msg.message)

#### Vortex → blow-up frames

Each panel is one callback snapshot. Early frames show the lid-driven vortex; later frames show the field losing structure as `max|u|` takes off. The returned fields after divergence are the last **finite** state, so the final panel is the picture just before the crash.

In [ ]:
# Pick a few frames spaced through the run, always including the last one.
idx = np.unique(np.linspace(0, len(snapshots) - 1, num=min(6, len(snapshots)), dtype=int))
fig, axes = plt.subplots(2, len(idx), figsize=(3.2 * len(idx), 6.5), dpi=100)

for col, i in enumerate(idx):
    s = snapshots[i]
    axes[0, col].contour(Xb, Yb, s["STREAMLINE"], 30)
    axes[0, col].set_title(f"t = {s['t']:.2f}\nmax|u| = {s['umax']:.2g}")
    axes[0, col].set_aspect("equal")
    axes[0, col].set_xticks([])
    axes[0, col].set_yticks([])

    im = axes[1, col].contourf(np.rot90(np.fliplr(s["vorticity"])), 30)
    axes[1, col].set_title(f"vort max = {s['vort_max']:.2g}")
    axes[1, col].set_aspect("equal")
    axes[1, col].set_xticks([])
    axes[1, col].set_yticks([])

axes[0, 0].set_ylabel("streamlines")
axes[1, 0].set_ylabel("vorticity")
fig.suptitle("Blow-up run: cavity vortex forming, then failing", y=1.02)
plt.tight_layout()

#### Growth of the singularity signal

In the claimed 3-D continuum constructions the gradient blows up like a negative power of $(T-t)$. Here we only plot the discrete diagnostics: `max|u|` and the pressure-iteration RMSE. Both stay moderate while the vortex is healthy, then surge at the diverged step.

In [ ]:
steps = np.arange(1, len(diag_b["umax"]) + 1)
t = steps * dt_blow
T_star = diag_b["diverged_time"]

fig, ax = plt.subplots(1, 2, figsize=(12, 4), dpi=100)
ax[0].semilogy(t, diag_b["umax"], "o-", ms=3)
ax[0].axvline(T_star, color="C3", ls="--", label=f"diverged at t = {T_star:.3g}")
ax[0].set_xlabel("t")
ax[0].set_ylabel("max |u|")
ax[0].set_title("Velocity blow-up")
ax[0].legend()

ax[1].semilogy(t, np.maximum(diag_b["error"], 1e-16), "o-", ms=3, color="C1")
ax[1].axvline(T_star, color="C3", ls="--")
ax[1].set_xlabel("t")
ax[1].set_ylabel("pressure RMSE")
ax[1].set_title("Pressure iteration residual")
plt.tight_layout()

### Side-by-side summary

Same physics parameters; only `dt` (and therefore the CFL) changes. The stable run keeps a bounded vortex. The aggressive run crashes in finite time and reports `diverged_step`.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5), dpi=100)

ax[0].contour(X, Y, STREAMLINE, 40)
ax[0].set_title(f"Stable  (CFL = {diag_s['cfl']:.2f}, dt = {dt_stable})")
ax[0].set_aspect("equal")

ax[1].contour(Xb, Yb, STREAMb, 40)
ax[1].set_title(
    f"Blow-up (CFL = {diag_b['cfl']:.2f}, dt = {dt_blow})\n"
    f"stopped at step {diag_b['diverged_step']}, t = {diag_b['diverged_time']:.3g}"
)
ax[1].set_aspect("equal")
plt.tight_layout()

print("stable : diverged_step =", diag_s["diverged_step"], " final max|u| =", f"{diag_s['umax'][-1]:.3g}")
print("blow-up: diverged_step =", diag_b["diverged_step"], " last max|u|  =", f"{diag_b['umax'][-1]:.3g}")

### Takeaway

1. The Millennium question is about **3-D continuum** Navier–Stokes. The recent Alpöge–Buckmaster claim (Claude-assisted) is a construction in that setting, not a lid-driven cavity run.
2. This notebook is a **2-D discrete** demonstration: an explicit scheme with CFL $> 1$ lets a lid-driven vortex grow and then blow up in finite simulation time.
3. The repository already detects that failure (`diverged_step`, last finite contours). No change to the solver is required for the demo.

For the mathematical claim itself, see the authors' paper and repository: [tristanbuckmaster/navier-stokes](https://github.com/tristanbuckmaster/navier-stokes).